# NIST RAG 시스템 구축

## 목표
- NIST PDF 3개를 Pinecone 벡터 DB에 저장
- 문서 근거를 보여주는 Q&A RAG 체인 구축
- LangSmith로 실행 로그 추적

## 사용 문서
1. NIST AI Risk Management Framework (AI RMF)
2. NIST Cybersecurity Framework 2.0 (CSF 2.0)
3. NIST Zero Trust Architecture (Zero Trust)


## 1. 환경 설정 및 패키지 설치


In [5]:
# 필요한 패키지 설치 (주석을 해제하여 실행)
%pip install -q langchain langchain-openai langchain-text-splitters langgraph langsmith pinecone-client pypdf python-dotenv tiktoken langchain-community langchain-core langchain_pinecone



[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
from pathlib import Path
from dotenv import load_dotenv
from typing import List, Dict

# LangChain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Pinecone
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

print("✅ 패키지 임포트 완료")


✅ 패키지 임포트 완료


In [7]:
# 환경 변수 로드
load_dotenv()

# 환경 변수 확인
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT")

# LangSmith 설정
os.environ["LANGSMITH_TRACING"] = "true"

# 확인
print(f"OPENAI_API_KEY 설정: {bool(OPENAI_API_KEY)}")
print(f"PINECONE_API_KEY 설정: {bool(PINECONE_API_KEY)}")
print(f"LANGSMITH_API_KEY 설정: {bool(LANGSMITH_API_KEY)}")
print(f"LangSmith 프로젝트: {os.getenv('LANGSMITH_PROJECT')}")


OPENAI_API_KEY 설정: True
PINECONE_API_KEY 설정: True
LANGSMITH_API_KEY 설정: True
LangSmith 프로젝트: pr-candid-waterspout-31


## 2. PDF 문서 로드 및 전처리


In [8]:
# PDF 파일 경로 설정
nist_dir = Path("nist")

pdf_files = [
    {
        "path": nist_dir / "nist_ai_risk_framework.pdf",
        "source": "ai_rmf",
        "doc_title": "NIST AI Risk Management Framework"
    },
    {
        "path": nist_dir / "nist_cybersecurity_framework.pdf",
        "source": "csf_2_0",
        "doc_title": "NIST Cybersecurity Framework 2.0"
    },
    {
        "path": nist_dir / "nist_zero_trust.pdf",
        "source": "zero_trust",
        "doc_title": "NIST Zero Trust Architecture"
    }
]

print("📁 PDF 파일 목록:")
for pdf in pdf_files:
    exists = pdf["path"].exists()
    status = "✅" if exists else "❌"
    print(f"{status} {pdf['doc_title']}")


📁 PDF 파일 목록:
✅ NIST AI Risk Management Framework
✅ NIST Cybersecurity Framework 2.0
✅ NIST Zero Trust Architecture


In [9]:
# PDF 문서 로드
all_documents = []

for pdf_info in pdf_files:
    print(f"\n📖 로딩 중: {pdf_info['doc_title']}...")
    
    loader = PyPDFLoader(str(pdf_info["path"]))
    documents = loader.load()
    
    # 메타데이터 추가
    for doc in documents:
        doc.metadata["source"] = pdf_info["source"]
        doc.metadata["doc_title"] = pdf_info["doc_title"]
        doc.metadata["page"] = doc.metadata.get("page", 0)
    
    all_documents.extend(documents)
    print(f"  ✅ {len(documents)}페이지 로드 완료")

print(f"\n📊 전체 로드된 페이지 수: {len(all_documents)}")



📖 로딩 중: NIST AI Risk Management Framework...
  ✅ 48페이지 로드 완료

📖 로딩 중: NIST Cybersecurity Framework 2.0...
  ✅ 32페이지 로드 완료

📖 로딩 중: NIST Zero Trust Architecture...
  ✅ 59페이지 로드 완료

📊 전체 로드된 페이지 수: 139


## 3. 문서 청킹 (Chunking)


In [10]:
# 텍스트 분할기 설정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,  # 각 청크의 최대 크기
    chunk_overlap=100,  # 청크 간 겹치는 부분
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# 문서 청킹
chunks = text_splitter.split_documents(all_documents)

print(f"✅ 총 {len(chunks)}개의 청크 생성")
print(f"\n첫 번째 청크 예시:")
print(f"내용: {chunks[0].page_content[:200]}...")
print(f"메타데이터: {chunks[0].metadata}")


✅ 총 535개의 청크 생성

첫 번째 청크 예시:
내용: NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)...
메타데이터: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-01-24T14:45:46-05:00', 'author': 'National Institute of Standards and Technology', 'keywords': 'Artificial Intelligence (AI); AI; AI RMF; AI RMF 1.0; AI systems; trustworthy and responsible AI.', 'moddate': '2025-06-04T13:01:45-04:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': 'As directed by the National Artificial Intelligence Initiative Act of 2020 (P.L. 116-283), the goal of the AI RMF is to offer a resource to the organizations designing, developing, deploying, or using AI systems to help manage the many risks of AI and promote trustworthy and responsible development and use of AI systems. The Framework is intended to be voluntary, rights-preserving, non-sector specific, and use-case agnostic, providi

In [11]:
# 문서별 청크 분포 확인
from collections import Counter

source_counts = Counter([chunk.metadata["source"] for chunk in chunks])

print("📊 문서별 청크 분포:")
for source, count in source_counts.items():
    doc_title = next(pdf["doc_title"] for pdf in pdf_files if pdf["source"] == source)
    print(f"  - {doc_title}: {count}개")


📊 문서별 청크 분포:
  - NIST AI Risk Management Framework: 171개
  - NIST Cybersecurity Framework 2.0: 111개
  - NIST Zero Trust Architecture: 253개


## 4. Pinecone 인덱스 생성 및 업서트


In [12]:
# Pinecone 클라이언트 초기화
pc = Pinecone(api_key=PINECONE_API_KEY)

# 인덱스 이름 설정
index_name = "nist-rag-index"

# 임베딩 모델 설정 (OpenAI text-embedding-3-small, dimension=1536)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

print(f"✅ Pinecone 클라이언트 초기화 완료")
print(f"📌 인덱스 이름: {index_name}")


✅ Pinecone 클라이언트 초기화 완료
📌 인덱스 이름: nist-rag-index


In [13]:
# 기존 인덱스 확인 및 생성
existing_indexes = [index.name for index in pc.list_indexes()]

if index_name in existing_indexes:
    print(f"⚠️  인덱스 '{index_name}'가 이미 존재합니다.")
    print(f"기존 인덱스를 삭제하고 새로 생성하려면 아래 주석을 해제하세요.")
    # pc.delete_index(index_name)
    # print(f"🗑️  기존 인덱스 삭제 완료")
else:
    print(f"🆕 새 인덱스 생성 중...")
    pc.create_index(
        name=index_name,
        dimension=1536,  # text-embedding-3-small 차원
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"✅ 인덱스 생성 완료")

# 인덱스 정보 출력
print(f"\n📊 현재 Pinecone 인덱스 목록: {pc.list_indexes().names()}")


🆕 새 인덱스 생성 중...
✅ 인덱스 생성 완료

📊 현재 Pinecone 인덱스 목록: ['llmops-rag-agent', 'rag-assignment-korquad', 'paper-reading-agent', 'rag-korquad-demo', 'nist-rag-index']


In [14]:
# Pinecone VectorStore에 문서 업서트
print(f"⏳ {len(chunks)}개 청크를 Pinecone에 업로드 중...")
print("(시간이 다소 걸릴 수 있습니다)")

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=index_name
)

print(f"✅ 벡터 업로드 완료!")


⏳ 535개 청크를 Pinecone에 업로드 중...
(시간이 다소 걸릴 수 있습니다)
✅ 벡터 업로드 완료!


In [15]:
# 기본 검색 테스트
test_query = "AI 리스크 관리란 무엇인가?"
results = vectorstore.similarity_search(test_query, k=3)

print(f"🔍 테스트 질문: '{test_query}'")
print(f"\n검색 결과 (상위 3개):")
for i, doc in enumerate(results, 1):
    print(f"\n[{i}] {doc.metadata['doc_title']} (페이지 {doc.metadata['page']})")
    print(f"내용: {doc.page_content[:150]}...")


🔍 테스트 질문: 'AI 리스크 관리란 무엇인가?'

검색 결과 (상위 3개):

[1] NIST AI Risk Management Framework (페이지 5.0)
내용: These risks make AI a uniquely challenging technology to deploy and utilize both for orga-
nizations and within society. Without proper controls, AI s...

[2] NIST AI Risk Management Framework (페이지 36.0)
내용: NIST AI 100-1 AI RMF 1.0
Practices related to managing AI risks are described in the NIST AI RMF Playbook. Table
4 lists the MANAGE function’s categor...

[3] NIST AI Risk Management Framework (페이지 13.0)
내용: Identifying and managing AI risks and potential impacts – both positive and negative – re-
quires a broad set of perspectives and actors across the AI...


## 5. RAG Q&A 체인 구축


In [16]:
# Retriever 생성 (k=4 청크 검색)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Retriever 생성 완료")


✅ Retriever 생성 완료


In [17]:
# RAG 프롬프트 템플릿
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 NIST(미국 국립표준기술연구소) 문서를 기반으로 답변하는 보안 및 AI 전문가입니다.

제공된 컨텍스트를 바탕으로 사용자의 질문에 정확하고 명확하게 답변하세요.

답변 시 반드시:
1. 어떤 NIST 문서를 참고했는지 명시하세요
2. 가능하면 문서의 어느 부분(페이지/섹션)을 참고했는지 언급하세요
3. 근거가 부족하면 "제공된 문서에서 관련 정보를 찾을 수 없습니다"라고 솔직하게 답하세요

컨텍스트:
{context}
"""),
    ("human", "{question}")
])

print("✅ RAG 프롬프트 템플릿 생성 완료")


✅ RAG 프롬프트 템플릿 생성 완료


In [18]:
# LLM 설정
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
    openai_api_key=OPENAI_API_KEY
)

# RAG 체인 구성
def format_docs(docs):
    """검색된 문서를 포맷팅"""
    formatted = []
    for i, doc in enumerate(docs, 1):
        formatted.append(
            f"[문서 {i}] {doc.metadata['doc_title']} (페이지 {doc.metadata['page']})\n"
            f"{doc.page_content}\n"
        )
    return "\n\n".join(formatted)

# 체인 조립
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# LangSmith 태그 추가
rag_chain = rag_chain.with_config({
    "tags": ["nist", "rag", "qa"],
    "run_name": "nist_rag_qa_chain"
})

print("✅ RAG 체인 구축 완료")
print("   - Retriever: Pinecone (k=4)")
print("   - LLM: gpt-4o-mini")
print("   - LangSmith 태그: nist, rag, qa")


✅ RAG 체인 구축 완료
   - Retriever: Pinecone (k=4)
   - LLM: gpt-4o-mini
   - LangSmith 태그: nist, rag, qa


## 6. RAG 시스템 테스트 (5개 이상 질문)


In [19]:
# 질문 1: AI Risk Management Framework 관련
question1 = "AI 리스크 관리 프레임워크(AI RMF)의 주요 목적은 무엇인가요?"

print(f"❓ 질문: {question1}\n")
answer1 = rag_chain.invoke(question1)
print(f"💡 답변:\n{answer1}")


❓ 질문: AI 리스크 관리 프레임워크(AI RMF)의 주요 목적은 무엇인가요?



/opt/homebrew/lib/python3.10/site-packages/pydantic/v1/main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


💡 답변:
AI 리스크 관리 프레임워크(AI RMF)의 주요 목적은 AI 리스크를 관리하고 신뢰할 수 있는 AI 시스템을 책임감 있게 개발하기 위한 대화, 이해 및 활동을 가능하게 하는 것입니다. 이 프레임워크는 다음과 같은 핵심 기능으로 구성되어 있습니다: GOVERN, MAP, MEASURE, 그리고 MANAGE. 각 기능은 카테고리와 하위 카테고리로 나뉘며, 특정 행동과 결과로 세분화됩니다. 이러한 구조는 조직 내에서 AI 리스크를 효과적으로 관리하고 소통할 수 있도록 돕습니다.

이 정보는 NIST AI Risk Management Framework 문서의 페이지 24.0에서 확인할 수 있습니다.


In [20]:
# 질문 2: AI RMF 관련
question2 = "NIST AI RMF의 4가지 핵심 기능(Functions)은 무엇인가요?"

print(f"❓ 질문: {question2}\n")
answer2 = rag_chain.invoke(question2)
print(f"💡 답변:\n{answer2}")


❓ 질문: NIST AI RMF의 4가지 핵심 기능(Functions)은 무엇인가요?

💡 답변:
NIST AI Risk Management Framework (AI RMF)에서 정의된 4가지 핵심 기능(Functions)은 다음과 같습니다:

1. **GOVERN**: AI 위험 관리 프로세스와 절차의 모든 단계에 적용됩니다.
2. **MAP**: AI 시스템의 특정 맥락과 AI 생애 주기의 특정 단계에서 적용됩니다.
3. **MEASURE**: AI 시스템의 특정 맥락과 AI 생애 주기의 특정 단계에서 적용됩니다.
4. **MANAGE**: AI 시스템의 특정 맥락과 AI 생애 주기의 특정 단계에서 적용됩니다.

이 정보는 NIST AI Risk Management Framework 문서의 페이지 7.0(문서 2)에서 확인할 수 있습니다.


In [21]:
# 질문 3: Cybersecurity Framework 관련
question3 = "NIST Cybersecurity Framework 2.0의 핵심 구성 요소는 무엇인가요?"

print(f"❓ 질문: {question3}\n")
answer3 = rag_chain.invoke(question3)
print(f"💡 답변:\n{answer3}")


❓ 질문: NIST Cybersecurity Framework 2.0의 핵심 구성 요소는 무엇인가요?

💡 답변:
NIST Cybersecurity Framework 2.0의 핵심 구성 요소는 CSF Core로, 이는 고수준의 사이버 보안 결과를 분류한 세 가지 주요 요소로 구성됩니다. 이 요소들은 다음과 같습니다:

1. **Functions (기능)**: 사이버 보안 관리의 주요 영역을 나타냅니다.
2. **Categories (범주)**: 각 기능 내에서 특정한 사이버 보안 결과를 세분화합니다.
3. **Subcategories (하위 범주)**: 각 범주에 대한 구체적인 결과를 정의합니다.

이러한 구성 요소는 조직이 사이버 보안 위험을 관리하는 데 도움을 주기 위해 설계되었으며, 각 조직의 필요에 따라 다르게 적용될 수 있습니다. 이 정보는 NIST Cybersecurity Framework 2.0 문서의 1페이지(섹션 1)에서 확인할 수 있습니다.


In [22]:
# 질문 4: Zero Trust 관련
question4 = "Zero Trust Architecture의 핵심 원칙은 무엇인가요?"

print(f"❓ 질문: {question4}\n")
answer4 = rag_chain.invoke(question4)
print(f"💡 답변:\n{answer4}")


❓ 질문: Zero Trust Architecture의 핵심 원칙은 무엇인가요?

💡 답변:
Zero Trust Architecture의 핵심 원칙은 다음과 같습니다:

1. **모든 데이터 소스와 컴퓨팅 서비스는 자원으로 간주**: 네트워크는 다양한 클래스의 장치로 구성될 수 있으며, 개인 소유 장치도 기업 소유 자원에 접근할 수 있는 경우 자원으로 분류될 수 있습니다.
   
2. **신뢰는 암묵적으로 부여되지 않음**: 신뢰는 지속적으로 평가되어야 하며, 자원에 대한 접근은 최소한의 권한만 부여되어야 합니다.

3. **최소 권한 원칙**: 사용자는 필요한 최소한의 권한만을 부여받아야 하며, 이는 읽기, 쓰기, 삭제 등의 권한을 포함합니다.

이 원칙들은 NIST SP 800-207 문서의 2.1 섹션(페이지 14.0)에서 설명되고 있습니다. Zero Trust Architecture는 이러한 원칙을 기반으로 하여 설계되고 배포됩니다.


In [23]:
# 질문 5: Zero Trust 관련
question5 = "Zero Trust 모델에서 마이크로 세그멘테이션(Micro-segmentation)은 어떤 역할을 하나요?"

print(f"❓ 질문: {question5}\n")
answer5 = rag_chain.invoke(question5)
print(f"💡 답변:\n{answer5}")


❓ 질문: Zero Trust 모델에서 마이크로 세그멘테이션(Micro-segmentation)은 어떤 역할을 하나요?

💡 답변:
Zero Trust 모델에서 마이크로 세그멘테이션(Micro-segmentation)은 네트워크 내의 개별 자원이나 자원 그룹을 고유한 네트워크 세그먼트에 배치하여 보호하는 역할을 합니다. 이 접근 방식은 각 자원 또는 관련 자원 그룹을 보호하기 위해 게이트웨이 보안 구성 요소를 사용하는 것을 포함합니다. 예를 들어, 지능형 스위치, 라우터, 차세대 방화벽(NGFW) 또는 특별 목적의 게이트웨이 장치를 사용하여 각 자원에 대한 정책 집행 포인트(PEP)를 설정할 수 있습니다.

이 정보는 NIST Zero Trust Architecture 문서의 3.1.2 섹션에서 확인할 수 있습니다. (페이지 20.0)


In [24]:
# 질문 6 (추가): 통합 질문
question6 = "AI 시스템의 보안을 강화하기 위해 Zero Trust 원칙을 어떻게 적용할 수 있나요?"

print(f"❓ 질문: {question6}\n")
answer6 = rag_chain.invoke(question6)
print(f"💡 답변:\n{answer6}")


❓ 질문: AI 시스템의 보안을 강화하기 위해 Zero Trust 원칙을 어떻게 적용할 수 있나요?

💡 답변:
AI 시스템의 보안을 강화하기 위해 Zero Trust 원칙을 적용하는 방법에 대해 설명하겠습니다. 이 내용은 NIST SP 800-207 "Zero Trust Architecture" 문서를 기반으로 합니다.

1. **신뢰의 지속적인 평가**: Zero Trust는 신뢰를 암묵적으로 부여하지 않고, 모든 접근 요청에 대해 지속적으로 인증 및 권한 부여를 수행해야 합니다. AI 시스템에 접근하는 모든 사용자와 시스템의 신원을 확인하고, 그들의 보안 상태를 평가해야 합니다 (문서 2, 페이지 12.0).

2. **최소 권한 원칙**: AI 시스템에 대한 접근은 필요한 최소한의 권한만 부여해야 합니다. 예를 들어, 데이터 읽기, 쓰기, 삭제 권한을 사용자나 시스템의 필요에 따라 제한해야 합니다 (문서 2, 페이지 12.0).

3. **지속적인 모니터링**: AI 시스템의 모든 활동을 지속적으로 모니터링하여 비정상적인 행동이나 위협을 신속하게 탐지하고 대응할 수 있어야 합니다. 이는 Zero Trust 아키텍처의 핵심 요소 중 하나입니다 (문서 1, 페이지 36.0).

4. **정책 기반 접근 제어**: AI 시스템의 접근은 정책 엔진을 통해 관리되어야 하며, 모든 통신은 사전에 정의된 정책에 따라 승인되어야 합니다. 이를 통해 시스템의 보안을 강화할 수 있습니다 (문서 1, 페이지 36.0).

5. **위험 분석 및 평가**: AI 시스템의 자산과 비즈니스 기능에 대한 위험을 분석하고 평가하여, 적절한 보호 조치를 시행해야 합니다. Zero Trust 아키텍처에서는 이러한 위험 분석이 필수적입니다 (문서 4, 페이지 9.0).

이러한 원칙들을 통해 AI 시스템의 보안을 강화하고, 사이버 공격에 대한 저항력을 높일 수 있습니다.


In [25]:
# 완료 요약
print("="*60)
print("🎉 NIST RAG 시스템 구축 완료!")
print("="*60)
print(f"\n📊 시스템 구성:")
print(f"  - 벡터 DB: Pinecone (인덱스: {index_name})")
print(f"  - 임베딩: OpenAI text-embedding-3-small")
print(f"  - LLM: gpt-4o-mini")
print(f"  - 총 청크 수: {len(chunks)}개")
print(f"  - 검색 청크 수(k): 4개")
print(f"\n✅ 6개 질문 테스트 완료")
print(f"✅ LangSmith 트레이싱 활성화됨")
print(f"\n다음 단계: 02_nist_rag_agent.ipynb로 이동하여 Agent 구축")


🎉 NIST RAG 시스템 구축 완료!

📊 시스템 구성:
  - 벡터 DB: Pinecone (인덱스: nist-rag-index)
  - 임베딩: OpenAI text-embedding-3-small
  - LLM: gpt-4o-mini
  - 총 청크 수: 535개
  - 검색 청크 수(k): 4개

✅ 6개 질문 테스트 완료
✅ LangSmith 트레이싱 활성화됨

다음 단계: 02_nist_rag_agent.ipynb로 이동하여 Agent 구축
